# Per-object Kerchunk Parquet Generation

Generates one parquet reference directory per new or changed source object.
Ledger is commited only if all generation tasks succeed

In [ ]:
%run .01.1-S3_inventory_diff

In [ ]:
import sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline.generate_parquet import parallelise_generate_references, save_ledger_after_success

import virtualizarr
import obstore
import obspec_utils
print("Imports OK")

In [ ]:
multithread_ref_gen= parallelise_generate_references(
    kp= kp,
    access_key= ACCESS_KEY,
    secret_key= SECRET_KEY,
    ledger_diff= ledger_diff,
    current_objects= ledger_current_objects
)

print('Phase 4 summary:', multithread_ref_gen['summary'])
print('Generated sample:', [r['key'] for r in multithread_ref_gen['results'] if r['status'] == 'generated'][:10])
print('Failures:', multithread_ref_gen['failures'][:5])

## Save to MasterLedger

Depends if errors found

In [ ]:
if multithread_ref_gen['summary']['failed'] == 0:
    save_ledger_after_success(
        ledger_path=kp['output']['ledger_path'],
        next_ledger= next_ledger,
        generation_summary= multithread_ref_gen['summary'],
    )
    print('Ledger committed:', kp['output']['ledger_path'])

else:
    print('Ledger NOT committed due to generation failures.')
